In [25]:
import os
from pathlib import Path
repo = Path.home() / "efficientml-lab"          # laptop; pod override below if needed
if Path("/workspace/qwenefficientai").exists():
    repo = Path("/workspace/qwenefficientai")
    os.environ["HF_HOME"] = "/workspace/hf_cache"
os.chdir(repo)
print(f"cwd: {os.getcwd()}")

cwd: /workspace/qwenefficientai


In [21]:
BACKEND = "llamacpp"        # "hf" on the pod, "llamacpp" on the laptop

if BACKEND == "hf":
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    model_id = "Qwen/Qwen3-4B-Instruct-2507"
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, device_map="cuda")
    model.eval()

In [22]:
import json, re
from ddgs import DDGS
import trafilatura
import requests

LLAMA_URL = "http://localhost:8080/v1/chat/completions"

def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '(30.19-28.5)/30.19*100'."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {e}"

def read_results_csv(query: str = "") -> str:
    """Return the contents of results/results.csv."""
    try:
        return open("results/results.csv").read()
    except FileNotFoundError:
        return "error: file not found"

TOOLS = {"calculator": calculator, "read_results_csv": read_results_csv}

tool_schemas = [
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression.",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string"}}, "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "read_results_csv",
        "description": "Read the experiment results CSV (config name, perplexity, tokens/sec).",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": []}}},
]

def parse_tool_calls(text):
    calls = []
    for m in re.findall(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", text, re.DOTALL):
        try:
            calls.append(json.loads(m))
        except json.JSONDecodeError:
            calls.append(None)  # malformed call — worth counting later
    return calls

def web_search(query: str, max_results: int = 5) -> str:
    """Search the web, return titles/URLs/snippets."""
    try:
        results = DDGS().text(query, max_results=max_results)
        return "\n\n".join(
            f"[{i+1}] {r['title']}\n{r['href']}\n{r['body']}"
            for i, r in enumerate(results)
        ) or "no results"
    except Exception as e:
        return f"search error: {e}"

def fetch_page(url: str) -> str:
    """Fetch a web page and return its main text (truncated)."""
    try:
        html = trafilatura.fetch_url(url)
        text = trafilatura.extract(html) or "could not extract text"
        return text[:4000]  # keep context manageable
    except Exception as e:
        return f"fetch error: {e}"

TOOLS.update({"web_search": web_search, "fetch_page": fetch_page})

tool_schemas += [
    {"type": "function", "function": {
        "name": "web_search",
        "description": "Search the web for current information. Returns titles, URLs, snippets.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"},
            "max_results": {"type": "integer"}}, "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "fetch_page",
        "description": "Fetch one URL from earlier search results and return its main text.",
        "parameters": {"type": "object", "properties": {
            "url": {"type": "string"}}, "required": ["url"]}}},
]

def generate_hf(messages):
    prompt = tok.apply_chat_template(
        messages, tools=tool_schemas, add_generation_prompt=True, tokenize=False)
    enc = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=2048, do_sample=False)
    return tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)

def generate_llamacpp(messages):
    r = requests.post(LLAMA_URL, json={
        "messages": messages,
        "tools": tool_schemas,        # --jinja renders these into Qwen's template
        "temperature": 0,             # = do_sample=False
        "max_tokens": 2048,
    }, timeout=600)                   # CPU is slow; don't let requests give up
    r.raise_for_status()
    msg = r.json()["choices"][0]["message"]
    text = msg.get("content") or ""
    # server parses Qwen tool calls into structured JSON; convert back to
    # <tool_call> text so parse_tool_calls & the loop stay backend-identical
    for tc in msg.get("tool_calls") or []:
        fn = tc["function"]
        args = fn.get("arguments") or "{}"
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except json.JSONDecodeError:
                args = {"_raw": args}
        text += ("\n<tool_call>\n"
                 + json.dumps({"name": fn["name"], "arguments": args})
                 + "\n</tool_call>")
    return text

GENERATE = {"hf": generate_hf, "llamacpp": generate_llamacpp}

def run_agent(user_msg, max_turns=10, verbose=True):
    messages = [{"role": "user", "content": user_msg}]
    for _ in range(max_turns):
        reply = GENERATE[BACKEND](messages)       
        if verbose:
            print("── model ──\n", reply.strip(), "\n")
        calls = parse_tool_calls(reply)
        if not calls:
            return reply
        messages.append({"role": "assistant", "content": reply})
        for call in calls:
            if call is None:
                result = "error: malformed tool call JSON"
            else:
                fn = TOOLS.get(call.get("name"))
                result = fn(**call.get("arguments", {})) if fn else "error: unknown tool"
            messages.append({"role": "tool", "content": str(result)})
    return "max turns exceeded"

In [23]:
#run_agent("Read the results CSV and tell me the baseline perplexity, "
          #"then compute what a 5% degradation from it would be.")

In [24]:
task = """Research the person named below using web_search (and fetch_page on the most
relevant result if snippets aren't enough). Then draft a short, warm professional
email introducing me (Ethan, a student working on LLM compression) and proposing a
chat. Rules:
- Only mention facts you actually found in the search results. If you can't verify
  something specific they've done, write a good email without fabricated details.
- Reference at most 1-2 specific things, naturally, not as a list of their resume.
- Under 150 words, no subject line fluff.

Person: Song Han, MIT
"""
_ = run_agent(task, max_turns=10)

ConnectionError: HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fd4685316d0>: Failed to establish a new connection: [Errno 111] Connection refused'))